# Transcriptor de YouTube

Genera transcripciones `.txt` y subtítulos `.srt` con **Whisper Large-v3 Turbo**.

Antes de empezar, elegí `Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU`. Usá solamente contenido propio o autorizado.

In [ ]:
# Instalación (una vez por sesión)
!pip -q install -U transformers accelerate yt-dlp sentencepiece ipywidgets
!apt-get -qq update && apt-get -qq install -y ffmpeg


In [ ]:
# Interfaz nativa de Jupyter / Google Colab (ipywidgets)
import re
import shutil
from pathlib import Path

import ipywidgets as widgets
import torch
from IPython.display import display, clear_output
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline

WORKDIR = Path('/content/whisper_resultados')
AUDIO_DIR = WORKDIR / 'audio'
OUT_DIR = WORKDIR / 'transcriptos'
ZIP_PATH = Path('/content/transcriptos_whisper.zip')
MODEL_ID = 'openai/whisper-large-v3-turbo'
MAX_URLS = 8
asr = None

def cargar_modelo():
    global asr
    if asr is not None:
        return
    if not torch.cuda.is_available():
        raise RuntimeError('No se detectó GPU. En Colab activá una GPU y ejecutá nuevamente esta celda.')
    modelo = AutoModelForSpeechSeq2Seq.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, low_cpu_mem_usage=True, use_safetensors=True
    ).to('cuda')
    procesador = AutoProcessor.from_pretrained(MODEL_ID)
    asr = pipeline(
        'automatic-speech-recognition', model=modelo, tokenizer=procesador.tokenizer,
        feature_extractor=procesador.feature_extractor, torch_dtype=torch.float16,
        device=0, chunk_length_s=30
    )

def seguro(nombre):
    return (re.sub(r'[^\w .-]+', '_', nombre, flags=re.UNICODE).strip()[:100] or 'video')

def formato_tiempo(segundos):
    ms = round((segundos - int(segundos)) * 1000)
    horas, resto = divmod(int(segundos), 3600)
    minutos, segundos = divmod(resto, 60)
    return f'{horas:02}:{minutos:02}:{segundos:02},{ms:03}'

def tiempo_legible(segundos):
    minutos, segundos = divmod(int(segundos or 0), 60)
    return f'{minutos} min {segundos:02} s' if minutos else f'{segundos} s'

def escribir_srt(chunks, destino):
    lineas = []
    for i, chunk in enumerate(chunks, 1):
        inicio, fin = chunk['timestamp']
        if inicio is None:
            continue
        if fin is None:
            fin = inicio + 2
        lineas.extend([str(i), f'{formato_tiempo(inicio)} --> {formato_tiempo(fin)}', chunk['text'].strip(), ''])
    destino.write_text('\n'.join(lineas), encoding='utf-8')

# --- Controles nativos ---
url_inputs, url_cards = [], []
lista_urls = widgets.VBox(layout=widgets.Layout(width='100%'))
estado = widgets.Label(value='Agregá una URL y presioná “Ver título” para comprobarla.')
progreso_total = widgets.FloatProgress(value=0, min=0, max=100, description='Total:', bar_style='', layout=widgets.Layout(width='100%', display='none'))
progresos_videos = widgets.VBox(layout=widgets.Layout(width='100%'))
idioma = widgets.Dropdown(options=['Detectar automáticamente', 'es', 'en', 'pt', 'fr', 'it', 'de'], value='es', description='Idioma:', layout=widgets.Layout(width='260px'))
agregar = widgets.Button(description='＋ Agregar otra URL', icon='plus', button_style='', layout=widgets.Layout(width='210px'))
procesar = widgets.Button(description='Comenzar a transcribir', icon='play', button_style='danger', layout=widgets.Layout(width='270px', height='42px'))
descargar = widgets.Button(description='Descargar ZIP con resultados', icon='download', button_style='success', layout=widgets.Layout(width='290px', height='42px', display='none'))

def crear_url():
    numero = len(url_inputs) + 1
    entrada = widgets.Text(placeholder='https://www.youtube.com/watch?v=…', description=f'Video {numero}:', layout=widgets.Layout(width='72%'))
    boton_titulo = widgets.Button(description='Ver título', icon='search', layout=widgets.Layout(width='120px'))
    titulo = widgets.Label(value='', layout=widgets.Layout(width='100%'))
    tarjeta = widgets.VBox([widgets.HBox([entrada, boton_titulo]), titulo], layout=widgets.Layout(border='1px solid #b8c3d6', border_radius='8px', padding='10px', margin='7px 0', width='100%'))
    def ver_titulo(_):
        if not entrada.value.strip():
            titulo.value = 'Pegá una URL primero.'
            return
        titulo.value = 'Buscando título…'
        try:
            import yt_dlp
            with yt_dlp.YoutubeDL({'quiet': True, 'noplaylist': True}) as ydl:
                info = ydl.extract_info(entrada.value.strip(), download=False)
            canal = info.get('channel') or info.get('uploader') or ''
            titulo.value = f'Encontrado: {info.get("title", "Video sin título")} — {canal} — {tiempo_legible(info.get("duration"))}'
        except Exception as e:
            titulo.value = f'No pude leer el enlace: {str(e)[:130]}'
    boton_titulo.on_click(ver_titulo)
    url_inputs.append(entrada)
    url_cards.append(tarjeta)
    lista_urls.children = tuple(url_cards)

def al_agregar(_):
    if len(url_inputs) < MAX_URLS:
        crear_url()
    if len(url_inputs) >= MAX_URLS:
        agregar.disabled = True
        agregar.description = f'Máximo: {MAX_URLS} videos'

def descargar_zip(_):
    if ZIP_PATH.exists():
        from google.colab import files
        files.download(str(ZIP_PATH))

def al_procesar(_):
    urls = [campo.value.strip() for campo in url_inputs if campo.value.strip()]
    if not urls:
        estado.value = 'Necesitás agregar al menos una URL.'
        return
    procesar.disabled = True
    agregar.disabled = True
    descargar.layout.display = 'none'
    progreso_total.value = 0
    progreso_total.layout.display = 'flex'
    barras = []
    for i in range(len(urls)):
        etiqueta = widgets.Label(value=f'Video {i+1}: en espera')
        barra = widgets.FloatProgress(value=0, min=0, max=100, layout=widgets.Layout(width='100%'))
        barras.append((etiqueta, barra))
    progresos_videos.children = tuple(widgets.VBox([etiqueta, barra], layout=widgets.Layout(border='1px solid #d8dee9', padding='8px', margin='5px 0')) for etiqueta, barra in barras)
    try:
        estado.value = 'Cargando Whisper Large-v3 Turbo…'
        cargar_modelo()
        if WORKDIR.exists():
            shutil.rmtree(WORKDIR)
        AUDIO_DIR.mkdir(parents=True)
        OUT_DIR.mkdir()
        import yt_dlp
        completados, errores = [], []
        for n, url in enumerate(urls, 1):
            etiqueta, barra = barras[n-1]
            try:
                etiqueta.value = f'Video {n}: descargando audio…'
                barra.value = 15
                with yt_dlp.YoutubeDL({'format': 'bestaudio/best', 'outtmpl': str(AUDIO_DIR / '%(id)s.%(ext)s'), 'quiet': True, 'noplaylist': True}) as ydl:
                    info = ydl.extract_info(url, download=True)
                    audio = Path(ydl.prepare_filename(info))
                titulo = seguro(info.get('title', f'video_{n}'))
                estimado = max(20, int((info.get('duration') or 300) * 0.10) + 12)
                etiqueta.value = f'{titulo}: transcribiendo (estimado: {tiempo_legible(estimado)})'
                barra.value = 50
                opciones = {'return_timestamps': True, 'generate_kwargs': {'task': 'transcribe'}}
                if idioma.value != 'Detectar automáticamente':
                    opciones['generate_kwargs']['language'] = idioma.value
                resultado = asr(str(audio), **opciones)
                (OUT_DIR / f'{titulo}.txt').write_text(resultado['text'].strip(), encoding='utf-8')
                escribir_srt(resultado.get('chunks', []), OUT_DIR / f'{titulo}.srt')
                etiqueta.value = f'{titulo}: listo'
                barra.value = 100
                barra.bar_style = 'success'
                completados.append(titulo)
            except Exception as e:
                etiqueta.value = f'Video {n}: no se pudo procesar ({str(e)[:90]})'
                barra.value = 100
                barra.bar_style = 'danger'
                errores.append(n)
            progreso_total.value = n / len(urls) * 100
        if not completados:
            estado.value = 'No se pudo transcribir ningún video. Revisá los enlaces.'
            return
        if ZIP_PATH.exists():
            ZIP_PATH.unlink()
        shutil.make_archive(str(ZIP_PATH.with_suffix('')), 'zip', OUT_DIR)
        estado.value = f'Listo: {len(completados)} video(s) procesado(s). El ZIP ya se puede descargar.'
        descargar.layout.display = 'flex'
    except Exception as e:
        estado.value = f'Error: {str(e)}'
    finally:
        procesar.disabled = False
        if len(url_inputs) < MAX_URLS:
            agregar.disabled = False

crear_url()
agregar.on_click(al_agregar)
procesar.on_click(al_procesar)
descargar.on_click(descargar_zip)

display(widgets.VBox([
    widgets.Label(value='TRANSCRIBIR VIDEOS DE YOUTUBE'),
    widgets.Label(value='Agregá una o varias URLs. Se crearán archivos TXT y SRT.'),
    lista_urls,
    agregar,
    widgets.HBox([idioma, procesar]),
    estado,
    progreso_total,
    progresos_videos,
    descargar
], layout=widgets.Layout(width='100%', max_width='900px', padding='10px')))
